In [1]:
!pip install torch torchvision torchaudio nltk numpy pandas tqdm matplotlib fastparquet huggingface_hub datasets tabulate

In [2]:
import sys
import os

# Path to the folder you want to add
subfolder_path = os.path.join(os.getcwd(), "n_gram_nn")

# Add it to sys.path
if subfolder_path not in sys.path:
    sys.path.append(subfolder_path)

In [ ]:
import torch

import torch.nn as nn
import torch.optim as optim
import pandas as pd
from tabulate import tabulate

from n_gram_nn.dataset import prepare_dataset_loaders, calculate_max_context_window
from n_gram_nn.model import SentenceModel
from n_gram_nn.train import *
from n_gram_nn.seeding import enforce_reproducibility
from n_gram_nn.visualisation import *
from n_gram_nn.tuning import calculate_possible_windows
from n_gram_nn.result_handling import best_configuration

result_folder = "n_gram_nn_results"

device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
    info_logger("CUDA detected")

/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to /Users/xk84vl/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt to /Users/xk84vl/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [4]:
### KEYS
REPLACE_FREQ_KEY = "replace_type"
TOP_FRACTION_KEY = "top_fraction"
EMBEDDING_DIM_KEY = "embedding_dim"
HIDDEN_DIM_KEY = "hidden_dim"
CONTEXT_WINDOW_KEY = "context_window_norm"
REPLACE_FRAC_KEY = "replace_frac"
LOSS_KEY = "loss"

In [5]:
# COMPARE RESULTS FROM FINETUNING ACROSS LANGUAGES TO FIND THE BEST COMBINATION
result_files = {
    "ko": "ko_tuning_results.csv",
    "ar": "ar_tuning_results.csv",
    "te": "te_tuning_results.csv"
}

best_configs, language_dfs = best_configuration(result_files, result_folder)

# # Sort by mean_loss
best_configs = best_configs.sort_values("mean_loss").head(5)

# Print best result per language
for language_df in language_dfs:
    best_row = language_df.nsmallest(1, LOSS_KEY).iloc[0]
    lang = best_row['language']
    loss = best_row[LOSS_KEY]
    print(f"The best loss for {lang} was {loss:.4f}")
print()

# Print the best results
print("=== Top 5 Cross-Language Configurations ===")
print(tabulate(best_configs, headers='keys', tablefmt='fancy_grid', showindex=False))

best_config = best_configs.head(1).iloc[0]  # This gives a Series instead of a dict

embed_dim = best_config[EMBEDDING_DIM_KEY]
hidden_dim = best_config[HIDDEN_DIM_KEY]
top_frac = best_config[TOP_FRACTION_KEY]
replace_type = best_config[REPLACE_FREQ_KEY]
replace_fraction = best_config[REPLACE_FRAC_KEY]
context_window_norm = best_config[CONTEXT_WINDOW_KEY]
batch_size = 16
epochs = 20

print("\n=== Best Model Configuration ===")
print(f"Embedding dimension       : {embed_dim}")
print(f"Hidden dimension          : {hidden_dim}")
print(f"Top fraction              : {top_frac}")
print(f"Replacement type          : {'Frequent Words' if replace_type else 'Infrequent words'}")
print(f"Replacement fraction      : {replace_fraction}")
print(f"Context window normalization: {context_window_norm}")
print("="*35 + "\n")


The best loss for ko was 3.5869
The best loss for ar was 4.3876
The best loss for te was 3.9611

=== Top 5 Cross-Language Configurations ===
╒════════════════╤════════════════╤═════════════════╤══════════════╤═══════════════════════╤════════════════╤═════════╤═════════╤═════════╤═════════════╤════════════╕
│ replace_type   │   top_fraction │   embedding_dim │   hidden_dim │ context_window_norm   │   replace_frac │      ar │      ko │      te │   mean_loss │   std_loss │
╞════════════════╪════════════════╪═════════════════╪══════════════╪═══════════════════════╪════════════════╪═════════╪═════════╪═════════╪═════════════╪════════════╡
│ False          │            0.1 │             256 │          512 │ max                   │           0.1  │ 4.38759 │ 3.58687 │ 3.96111 │     3.97852 │   0.400641 │
├────────────────┼────────────────┼─────────────────┼──────────────┼───────────────────────┼────────────────┼─────────┼─────────┼─────────┼─────────────┼────────────┤
│ False          │      

In [6]:
def get_context_window(language):
    context_window = calculate_max_context_window(language)
    if context_window_norm == "min":
        context_windows = calculate_possible_windows(context_window)
        context_window = min(window for window in context_windows if window > 1)
    return context_window

In [7]:
enforce_reproducibility(42)

language = "ko"
model_name = f"model_{language}.model"
context_window = get_context_window(language)

train_loader, val_loader, test_loader, vocab = prepare_dataset_loaders(language, top_frac, replace_type, batch_size, replace_fraction, context_window)

model = SentenceModel(len(vocab), embed_dim, hidden_dim, context_window).to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.NLLLoss()

val_losses, val_accs, val_pp, val_topk = train(model, device, train_loader, val_loader, optimizer, criterion, epochs)

plot_two_curves(val_losses, val_accs, save_path=f"{result_folder}/losses_{language}.pdf")
plot_one_curve(val_pp, "Validation Perplexity", "Perplexity", save_path=f"{result_folder}/perplexity_{language}.pdf")
plot_one_curve(val_topk, "Validation Top k Accuracy", "Accuracy", save_path=f"{result_folder}/accuracy_{language}.pdf")

torch.save(model.state_dict(), f"{result_folder}/{model_name}")

test_metrics = evaluate(model, device, test_loader, criterion, top_k=5)

[2025-10-24 10:56:02,054] - [INFO] - Train dataset contains 2183 samples
[2025-10-24 10:56:02,055] - [INFO] - Validation dataset contains 239 samples
[2025-10-24 10:56:02,055] - [INFO] - Test dataset contains 356 samples
[2025-10-24 10:56:02,056] - [INFO] - Top 10.00% frequent words (410) out of 4109 total words
[2025-10-24 10:56:02,056] - [INFO] - The fragment to replace is 3.148048780487805 of the 410 frequent words (410 out of 12907)
[2025-10-24 10:56:02,067] - [INFO] - Replaced 410 / 12907 tokens (3.18%)
[2025-10-24 10:56:02,076] - [INFO] - Replaced 13 / 1397 tokens (0.93%)
[2025-10-24 10:56:02,080] - [INFO] - Replaced 28 / 2088 tokens (1.34%)


100%|██████████| 20/20 [01:31<00:00,  4.59s/it]


[2025-10-24 10:57:34,695] - [INFO] - Eval Loss: 3.6664 | Perplexity: 39.11 | Accuracy: 39.08% | Top-5 Accuracy: 62.74%


In [15]:
enforce_reproducibility(42)

language = "ar"
model_name = f"model_{language}.model"
context_window = get_context_window(language)
epochs = 9

train_loader, val_loader, test_loader, vocab = prepare_dataset_loaders(language, top_frac, replace_type, batch_size, replace_fraction, context_window)

model = SentenceModel(len(vocab), embed_dim, hidden_dim, context_window).to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.NLLLoss()

val_losses, val_accs, val_pp, val_topk = train(model, device, train_loader, val_loader, optimizer, criterion, epochs)

plot_two_curves(val_losses, val_accs, save_path=f"{result_folder}/losses_{language}.pdf")
plot_one_curve(val_pp, "Validation Perplexity", "Perplexity", save_path=f"{result_folder}/perplexity_{language}.pdf")
plot_one_curve(val_topk, "Validation Top k Accuracy", "Accuracy", save_path=f"{result_folder}/accuracy_{language}.pdf")

torch.save(model.state_dict(), f"{result_folder}/{model_name}")

test_metrics = evaluate(model, device, test_loader, criterion, top_k=5)

[2025-10-24 11:47:40,196] - [INFO] - Train dataset contains 2309 samples
[2025-10-24 11:47:40,196] - [INFO] - Validation dataset contains 249 samples
[2025-10-24 11:47:40,196] - [INFO] - Test dataset contains 415 samples
[2025-10-24 11:47:40,198] - [INFO] - Top 10.00% frequent words (537) out of 5373 total words
[2025-10-24 11:47:40,199] - [INFO] - The fragment to replace is 2.920670391061453 of the 537 frequent words (537 out of 15684)
[2025-10-24 11:47:40,212] - [INFO] - Replaced 537 / 15684 tokens (3.42%)
[2025-10-24 11:47:40,223] - [INFO] - Replaced 31 / 1736 tokens (1.79%)
[2025-10-24 11:47:40,226] - [INFO] - Replaced 64 / 2814 tokens (2.27%)


100%|██████████| 9/9 [01:04<00:00,  7.21s/it]


[2025-10-24 11:48:45,551] - [INFO] - Eval Loss: 4.3591 | Perplexity: 78.18 | Accuracy: 26.37% | Top-5 Accuracy: 58.71%


In [9]:
enforce_reproducibility(42)

language = "te"
model_name = f"model_{language}.model"
context_window = get_context_window(language)

train_loader, val_loader, test_loader, vocab = prepare_dataset_loaders(language, top_frac, replace_type, batch_size, replace_fraction, context_window)

model = SentenceModel(len(vocab), embed_dim, hidden_dim, context_window).to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.NLLLoss()

val_losses, val_accs, val_pp, val_topk = train(model, device, train_loader, val_loader, optimizer, criterion, epochs)

plot_two_curves(val_losses, val_accs, save_path=f"{result_folder}/losses_{language}.pdf")
plot_one_curve(val_pp, "Validation Perplexity", "Perplexity", save_path=f"{result_folder}/perplexity_{language}.pdf")
plot_one_curve(val_topk, "Validation Top k Accuracy", "Accuracy", save_path=f"{result_folder}/accuracy_{language}.pdf")

torch.save(model.state_dict(), f"{result_folder}/{model_name}")

test_metrics = evaluate(model, device, test_loader, criterion, top_k=5)

[2025-10-24 11:00:26,557] - [INFO] - Train dataset contains 1224 samples
[2025-10-24 11:00:26,558] - [INFO] - Validation dataset contains 131 samples
[2025-10-24 11:00:26,558] - [INFO] - Test dataset contains 384 samples
[2025-10-24 11:00:26,559] - [INFO] - Top 10.00% frequent words (224) out of 2249 total words
[2025-10-24 11:00:26,559] - [INFO] - The fragment to replace is 3.644196428571429 of the 224 frequent words (224 out of 8163)
[2025-10-24 11:00:26,566] - [INFO] - Replaced 224 / 8163 tokens (2.74%)
[2025-10-24 11:00:26,572] - [INFO] - Replaced 11 / 885 tokens (1.24%)
[2025-10-24 11:00:26,575] - [INFO] - Replaced 29 / 2684 tokens (1.08%)


100%|██████████| 20/20 [00:42<00:00,  2.14s/it]


[2025-10-24 11:01:09,704] - [INFO] - Eval Loss: 3.7490 | Perplexity: 42.48 | Accuracy: 40.80% | Top-5 Accuracy: 60.10%


In [10]:
enforce_reproducibility(42)

language = "en"
model_name = f"model_{language}.model"
context_window = get_context_window(language) // 2

train_loader, val_loader, test_loader, vocab = prepare_dataset_loaders(language, top_frac, replace_type, batch_size, replace_fraction, context_window)

model = SentenceModel(len(vocab), embed_dim, hidden_dim, context_window).to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.NLLLoss()

val_losses, val_accs, val_pp, val_topk = train(model, device, train_loader, val_loader, optimizer, criterion, epochs)

plot_two_curves(val_losses, val_accs, save_path=f"{result_folder}/losses_{language}.pdf")
plot_one_curve(val_pp, "Validation Perplexity", "Perplexity", save_path=f"{result_folder}/perplexity_{language}.pdf")
plot_one_curve(val_topk, "Validation Top k Accuracy", "Accuracy", save_path=f"{result_folder}/accuracy_{language}.pdf")

torch.save(model.state_dict(), f"{result_folder}/{model_name}")

test_metrics = evaluate(model, device, test_loader, criterion, top_k=5)

[2025-10-24 11:01:21,578] - [INFO] - Train dataset contains 13798 samples
[2025-10-24 11:01:21,578] - [INFO] - Validation dataset contains 1545 samples
[2025-10-24 11:01:21,578] - [INFO] - Test dataset contains 3011 samples
[2025-10-24 11:01:21,711] - [INFO] - Top 10.00% frequent words (8732) out of 87331 total words
[2025-10-24 11:01:21,712] - [INFO] - The fragment to replace is 18.235868071461294 of the 8732 frequent words (8732 out of 1592356)
[2025-10-24 11:01:22,536] - [INFO] - Replaced 8732 / 1592356 tokens (0.55%)
[2025-10-24 11:01:28,433] - [INFO] - Replaced 543 / 179027 tokens (0.30%)
[2025-10-24 11:01:29,524] - [INFO] - Replaced 1011 / 351424 tokens (0.29%)


  0%|          | 0/20 [30:10<?, ?it/s]


KeyboardInterrupt: 